# Verificación de integridad y organización final - MMOTU / OTU_2D

Este notebook corresponde a la verificación final del conjunto de datos preparado y organizado mediante el pipeline. La verificación se centra en comprobar la consistencia entre los metadatos generados y los archivos almacenados físicamente.

> **Prerrequisitos.**  Requiere la salida de `03_ejecucion_pipeline.ipynb` en `data/processed/MMOTU_OTU_2D/` (`metadata.csv`, `exclusions.csv`, imágenes y máscaras). Para reproducir esa etapa desde cero es necesario disponer localmente del dataset fuente MMOTU / OTU_2D en `data/raw/MMOTU/OTU_2D/`. El dataset no se distribuye con el repositorio.
>

In [1]:
from pathlib import Path
import pandas as pd

def locate_project_root() -> Path:
    current_dir = Path.cwd().resolve()

    for candidate in [current_dir, *current_dir.parents]:
        if (
            (candidate / "src").exists()
            and (candidate / "data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "No se pudo localizar la raíz del proyecto."
    )


PROJECT_ROOT = locate_project_root()

PROCESSED_DATA_DIR = (PROJECT_ROOT / "data" / "processed" / "MMOTU_OTU_2D")
METADATA_FILE = PROCESSED_DATA_DIR / "metadata.csv"
EXCLUSIONS_FILE = PROCESSED_DATA_DIR / "exclusions.csv"

In [2]:
metadata_df = pd.read_csv(METADATA_FILE)
print("Registros cargados:", len(metadata_df))

Registros cargados: 1469


## 1. Verificación de la estructura de organización

Se comprueba la existencia de los directorios y archivos principales generados durante la ejecución del pipeline.

In [3]:
expected_paths = {
    "train_images": PROCESSED_DATA_DIR / "train" / "images",
    "train_masks": PROCESSED_DATA_DIR / "train" / "masks",
    "validation_images": PROCESSED_DATA_DIR / "validation" / "images",
    "validation_masks": PROCESSED_DATA_DIR / "validation" / "masks",
    "metadata": METADATA_FILE,
    "exclusions": EXCLUSIONS_FILE,
}

structure_results = {name: path.exists()
    for name, path in expected_paths.items()
}

structure_results

{'train_images': True,
 'train_masks': True,
 'validation_images': True,
 'validation_masks': True,
 'metadata': True,
 'exclusions': True}

In [4]:
# Verificación de la estructura del dataset
assert all(structure_results.values())
print("Estructura final verificada correctamente.")

Estructura final verificada correctamente.


## 2. Correspondencia entre metadatos y archivos organizados

Se verifica que cada muestra registrada como incluida en los metadatos disponga físicamente de su imagen y máscara correspondientes dentro del conjunto organizado.

In [5]:
# Función para resolver rutas relativas a la raíz del proyecto
def resolve_project_path(path_value):
    path = Path(path_value)

    if path.is_absolute():
        return path

    return PROJECT_ROOT / path

In [6]:
missing_metadata_images = []
missing_metadata_masks = []

for _, row in metadata_df.iterrows():
    image_path = resolve_project_path(row["organized_image_path"])
    mask_path = resolve_project_path(row["organized_mask_path"])

    if not image_path.exists():
        missing_metadata_images.append(row["image_id"])

    if not mask_path.exists():
        missing_metadata_masks.append(row["image_id"])

print("Imágenes registradas pero ausentes:", len(missing_metadata_images))
print("Máscaras registradas pero ausentes:", len(missing_metadata_masks))

Imágenes registradas pero ausentes: 0
Máscaras registradas pero ausentes: 0


In [7]:
assert len(missing_metadata_images) == 0
assert len(missing_metadata_masks) == 0
print("Todos los registros poseen sus archivos físicos correspondientes.")

Todos los registros poseen sus archivos físicos correspondientes.


## 3. Verificación de archivos no registrados

Se comprueba que no existan imágenes o máscaras almacenadas en el conjunto organizado que carezcan de un registro correspondiente en los metadatos.

In [ ]:
actual_train_images = {
    path.name
    for path in (PROCESSED_DATA_DIR / "train" / "images").iterdir()
    if path.is_file()
}

actual_train_masks = {
    path.name
    for path in (PROCESSED_DATA_DIR / "train" / "masks").iterdir()
    if path.is_file()
}

actual_validation_images = {
    path.name
    for path in (PROCESSED_DATA_DIR / "validation"/ "images").iterdir()
    if path.is_file()
}

actual_validation_masks = {
    path.name
    for path in (PROCESSED_DATA_DIR / "validation" / "masks").iterdir()
    if path.is_file()
}

In [ ]:
train_metadata = metadata_df[metadata_df["split"] == "train"]
validation_metadata = metadata_df[metadata_df["split"] == "validation"]

# Verificación de integridad de los archivos físicos con respecto a los registros en metadata.csv
expected_train_images = {
    Path(path).name
    for path in train_metadata["organized_image_path"]
}
expected_train_masks = {
    Path(path).name
    for path in train_metadata["organized_mask_path"]
}
expected_validation_images = {
    Path(path).name
    for path in validation_metadata["organized_image_path"]
}
expected_validation_masks = {
    Path(path).name
    for path in validation_metadata["organized_mask_path"]
}

In [10]:
# Comparación de los archivos físicos con los registros en metadata.csv
orphan_train_images = (actual_train_images - expected_train_images)
orphan_train_masks = (actual_train_masks - expected_train_masks)
orphan_validation_images = (actual_validation_images - expected_validation_images)
orphan_validation_masks = (actual_validation_masks - expected_validation_masks)

print("Imágenes huérfanas en train:", len(orphan_train_images))
print("Máscaras huérfanas en train:", len(orphan_train_masks))
print("Imágenes huérfanas en validation:", len(orphan_validation_images))
print("Máscaras huérfanas en validation:", len(orphan_validation_masks))

Imágenes huérfanas en train: 0
Máscaras huérfanas en train: 0
Imágenes huérfanas en validation: 0
Máscaras huérfanas en validation: 0


In [11]:
# Verificación de que no existan archivos huérfanos
assert len(orphan_train_images) == 0
assert len(orphan_train_masks) == 0
assert len(orphan_validation_images) == 0
assert len(orphan_validation_masks) == 0

print("No se encontraron archivos huérfanos.")

No se encontraron archivos huérfanos.


## 4. Verificación de completitud del conjunto

Se comprueba que la cantidad de muestras organizadas corresponda con la cantidad registrada en los metadatos y con las particiones originales de MMOTU utilizadas durante la ejecución del pipeline (`train` = 1000, `validation` = 469). La partición experimental `train` / `validation` / `test` (800 / 200 / 469) se verifica en `06_generacion_split_experimental.ipynb`.

In [12]:
final_counts = {
    "metadata_records": len(metadata_df),
    "train_images": len(actual_train_images),
    "train_masks": len(actual_train_masks),
    "validation_images": len(actual_validation_images),
    "validation_masks": len(actual_validation_masks),
}

final_counts

{'metadata_records': 1469,
 'train_images': 1000,
 'train_masks': 1000,
 'validation_images': 469,
 'validation_masks': 469}

In [13]:
assert final_counts["metadata_records"] == 1469
assert final_counts["train_images"] == 1000
assert final_counts["train_masks"] == 1000
assert final_counts["validation_images"] == 469
assert final_counts["validation_masks"] == 469

print("Completitud final verificada correctamente.")

Completitud final verificada correctamente.


## 5. Resumen de la verificación final

Los controles realizados permiten determinar el cumplimiento de las condiciones de integridad y organización establecidas para el conjunto preparado.

In [14]:
verification_summary = pd.DataFrame([
    {
        "verificacion":
            "Estructura de directorios y reportes",
        "resultado":
            "Completa",
        "estado":
            "Cumplido",
    },
    {
        "verificacion":
            "Registros con imagen física existente",
        "resultado":
            f"{len(metadata_df) - len(missing_metadata_images)}/{len(metadata_df)}",
        "estado":
            "Cumplido",
    },
    {
        "verificacion":
            "Registros con máscara física existente",
        "resultado":
            f"{len(metadata_df) - len(missing_metadata_masks)}/{len(metadata_df)}",
        "estado":
            "Cumplido",
    },
    {
        "verificacion":
            "Archivos huérfanos",
        "resultado":
            0,
        "estado":
            "Cumplido",
    },
    {
        "verificacion":
            "Muestras organizadas",
        "resultado":
            "1000 train + 469 validation = 1469",
        "estado":
            "Cumplido",
    },
])

verification_summary

,verificacion,resultado,estado
0,Estructura de directorios y reportes,Completa,Cumplido
1,Registros con imagen física existente,1469/1469,Cumplido
2,Registros con máscara física existente,1469/1469,Cumplido
3,Archivos huérfanos,0,Cumplido
4,Muestras organizadas,1000 train + 469 validation = 1469,Cumplido
